## **URL links for the Kobo form**

In [1]:
import requests
import os
from dotenv import load_dotenv

load_dotenv()

True

## Importing the modules required for this notebook

> Add blockquote



This notebook requires pandas, geopandas and follium for the data visualization & Analysis.

In [1]:
import requests
import pandas as pd
import os

#### Use this cell for installing the geopandas module
If you bear any problem while running the cells, Please install the required modules as per your need with similar way as of below:

In [2]:
import geopandas as gpd
from shapely import wkt

In [3]:
import numpy as np

In [4]:
import folium
import matplotlib.pyplot as plt

In [5]:
data_url = os.getenv('KOBO_DATA_URL')
username = os.getenv('KOBO_USERNAME')
password = os.getenv('KOBO_PASSWORD')


## Reading the data directly from Kobo server:

It will be accesing the real time data as of time of runing this code. But be careful of the fact that, Enumerators may have collected the data in their device and haven't yet submitted the data to server

In [10]:
rows = []
url = data_url

while url:
    r = requests.get(url, auth=(username, password))
    r.raise_for_status()
    j = r.json()
    rows.extend(j["results"])
    url = j["next"]

df = pd.json_normalize(rows)
print(len(df))   # 3031

3031


In [13]:
df.shape

(3031, 30)

In [14]:
df.columns

Index(['_id', 'formhub/uuid', 'deviceid', 'start', 'end', 'enumerators',
       'A/Plot_size', 'A/first_criteria/Nearby_plot',
       'A/first_criteria/second_criteria/location_info/state',
       'A/first_criteria/second_criteria/location_info/district',
       'A/first_criteria/second_criteria/B/crop',
       'A/first_criteria/second_criteria/C/image_widget_no_choose',
       'A/first_criteria/second_criteria/C/PlotGPS',
       'A/first_criteria/second_criteria/C/shape',
       'A/first_criteria/second_criteria/C/shape_area',
       'A/first_criteria/second_criteria/C/rounded_shape_area', '__version__',
       'meta/instanceID', 'meta/instanceName', '_xform_id_string', '_uuid',
       '_attachments', '_status', '_geolocation', '_submission_time',
       '_submitted_by', 'meta/rootUuid',
       'A/first_criteria/second_criteria/C/remarks',
       'A/first_criteria/second_criteria/B/specify_other_vegetable',
       'A/first_criteria/second_criteria/B/specify_other_crop'],
      dtype='

In [15]:
df["_geolocation"].head()

0      [28.66823, 80.4941262]
1     [28.668145, 80.4941967]
2      [28.6679111, 80.49249]
3    [28.6679167, 80.4925267]
4    [28.6683256, 80.4904738]
Name: _geolocation, dtype: object

In [17]:
col_rename = [
    'id', 'uid', 'deviceid', 'start', 'end', 'enumerators',
    'Plot_size', 'Nearby_plot',
    'state',
    'district',
    'crop',
    'image_widget_no_choose',
    'PlotGPS',
    'shape',
    'shape_area',
    'rounded_shape_area', 'version',
    'instanceID', 'instanceName', 'xform_id_string', 'uuid',
    'attachments', 'status', 'geolocation', 'submission_time',
    'submitted_by', 'rootUuid',
    'remarks',
    'specify_other_vegetable',
    'specify_other_crop'
]
df.columns = col_rename

## Data submitted by individual enumerators (Number of samples for each)

In [18]:
df.groupby(['enumerators'])['crop'].count()

enumerators
Krishna_Kafle         10
Saksham_Chaudhary    210
Sonam                 20
Stephanie             38
Subodh_Tiwari        203
anita                103
ashish               117
ashok                  1
asmita                 5
basanta_giri          47
daman                112
deb                   12
kanchan              210
khem_raj             191
mira                 158
pawan                283
pitamber             252
rabina               111
rabindra_rawat        24
ram                  120
sanat_kc              50
santosh              101
santu                110
saral_karki           10
sujan                227
sujan_subedi         193
umesh                113
Name: crop, dtype: int64

### Crop sample size for whole study area

In [19]:
df.groupby(['crop'])['crop'].count()

crop
Buckwheat               4
Mustard               212
Pasture                63
Potato                 85
Weedy_fallow_land     195
banana                 40
barley                  8
dry_fallow_land        74
gram                    9
grassland              32
lentil                129
linseed                13
maize                 258
mustard_lentil         68
other                 132
pea                     7
pigeon_pea             36
shrub_tree             44
sugarcane              69
vegetable             130
wheat                1367
wheat_mustard          56
Name: crop, dtype: int64

## Number of samples by each enumerators with crop type samples

In [20]:
df.groupby(['enumerators','crop'])['crop'].count().head()

enumerators        crop   
Krishna_Kafle      Potato      1
                   other       2
                   wheat       7
Saksham_Chaudhary  Mustard     2
                   Pasture    15
Name: crop, dtype: int64

## Each Region (AOI) wise crop samples collected so far:

#### Note: Achham data has to be discarded here

In [21]:
# Ref
# https://stackoverflow.com/questions/67093514/how-can-i-convert-a-geoshape-geotrace-geopoint-to-geojson

In [22]:
region_dist_dict = {
    '233': 'Birgunj',
    '234': 'Birgunj',
    '320': 'Kathmandu',
    '324': 'Kathmandu',
    '325': 'Kathmandu',
    '326': 'Kathmandu',
    '327': 'Kathmandu',
    '329': 'Kathmandu',
    '439': 'Pokhara',
    '440': 'Pokhara',
    '546': 'Kapilvastu',
    '550': 'Kapilvastu',
    '556': 'Dang',
    '557': 'NepalGunj',
    '558': 'NepalGunj',
    '577': 'Rukum',
    '654': 'Rukum',
    '769': 'Achham',
    '771': 'Dhangadi',
    '772': 'Dhangadi',
}
df_dict ={
    'district':list(region_dist_dict.keys()),
    'AOI':list(region_dist_dict.values())
}
df_reg = pd.DataFrame.from_dict(df_dict)
df = df.merge(df_reg, how='inner', on='district')

In [23]:
count_dist_crop = df.groupby(['AOI','crop','enumerators'])['crop'].count()
pd.set_option("display.max_rows", df.index.shape[0])
print(count_dist_crop)

AOI         crop               enumerators      
Achham      grassland          rabindra_rawat         2
            shrub_tree         rabindra_rawat         2
            wheat              mira                   1
                               rabindra_rawat        20
Birgunj     Mustard            Saksham_Chaudhary      2
            Pasture            Saksham_Chaudhary     15
                               Subodh_Tiwari         16
            dry_fallow_land    Saksham_Chaudhary      2
                               Subodh_Tiwari         19
            grassland          Saksham_Chaudhary      2
                               Subodh_Tiwari          3
                               sujan                  1
            lentil             Saksham_Chaudhary     17
                               Subodh_Tiwari         15
            maize              Saksham_Chaudhary     37
                               Subodh_Tiwari         33
                               sujan                  1

In [24]:
df.groupby(['AOI',])['crop'].count()

AOI
Achham         25
Birgunj       420
Dang          368
Dhangadi      435
Kapilvastu    295
Kathmandu     501
NepalGunj     489
Pokhara       388
Rukum          97
Name: crop, dtype: int64

## Point Visualization for monitoring

In [25]:
def wkt_point(pon):
    point = "POINT ({} {})".format(pon[1], pon[0])
    return point

In [26]:
df['point_geom'] = df['geolocation'].map(wkt_point)
# df

df['point_geometry'] = df.point_geom.apply(wkt.loads)
df.drop('point_geom', axis=1, inplace=True) #Drop WKT column


gdf_point = gpd.GeoDataFrame(df, geometry='point_geometry')
# gdf_point

# Monitoring the feild polygons for each crop

In [27]:
df['shape'][0]

'28.668157106680017 80.49440413713455 0.0 0.0;28.667538453556368 80.49417313188314 0.0 0.0;28.667639944690134 80.49387406557798 0.0 0.0;28.6681433 80.4942067 0.0 0.0;28.668157106680017 80.49440413713455 0.0 0.0'

In [28]:
sample = df['shape'][0]
# sample

In [29]:
from shapely.geometry import LineString, Point, Polygon
from shapely.wkt import dumps

In [30]:
# https://python.hotexamples.com/examples/shapely.wkt/-/dumps/python-dumps-function-examples.html#0x346e25d2de439ed401c723d3f6e3e4c911cb28698b845c5bec33796af1b082ac-106,,134,
# https://github.com/Cadasta/cadasta-platform/blob/master/cadasta/xforms/utils.py

def odk_geom_to_wkt(coords):
    """Convert geometries in ODK format to WKT."""

    if coords == '':
        return ''
#     print(coords)
    if str(coords)!='nan':
        coords = coords.replace('\n', '')
        coords = coords.split(';')
        coords = [c.strip() for c in coords]
        if (coords[-1] == ''):
            coords.pop()

        if len(coords) > 1:
            # check for a geoshape taking into account
            # the bug in odk where the second coordinate in a geoshape
            # is the same as the last (first and last should be equal)
            if len(coords) > 3:
                if coords[1] == coords[-1]:  # geom is closed
                    coords.pop()
                    coords.append(coords[0])
            points = []
            for coord in coords:
                coord = coord.split(' ')
                coord = [x for x in coord if x]
                latlng = [float(coord[1]),
                          float(coord[0])]
                points.append(tuple(latlng))
            if (coords[0] != coords[-1] or len(coords) == 2):
                return dumps(LineString(points))
            else:
                return dumps(Polygon(points))
        else:
            latlng = coords[0].split(' ')
            latlng = [x for x in latlng if x]
            return dumps(Point(float(latlng[1]), float(latlng[0])))
    else:
        return np.nan

In [31]:
# def split_colon(x):
#     a = x.split(';')

#     return

# def wkt_coverter(poly):
#     print(poly)
#     if str(poly)!='nan':
#         print(list(map(lambda x:x.replace(' 0.0 0.0', '').split(' '), poly.split(';'))))
#         polygon_coordinate = list(map(lambda z:"{} {}".format(z[1],z[0]), map(lambda y:y[:2], map(lambda x:x.replace(' 0.0 0.0', '').split(' '), poly.split(';')))))
#         polygon_coordinate
#         wkt_polygon = "POLYGON (({}))".format(", ".join(map(str, polygon_coordinate)))
#         return wkt_polygon
#     else:
#         return np.nan

def wkt_loads(x):
    try:
        return wkt.loads(x)
    except Exception:
        return None

In [32]:
df['geom_poly'] = df['shape'].map(odk_geom_to_wkt)
# df

In [33]:
df_new = df
df_new['geometry'] = df_new.geom_poly.apply(wkt_loads)
df_new.drop('geom_poly', axis=1, inplace=True) #Drop WKT column
# df_ex = df_new.isna(subset=['geometry'])
df_new = df_new.dropna(subset=['geometry'])

# Geopandas GeoDataFrame
gdf_poly = gpd.GeoDataFrame(df_new, geometry='geometry')

In [34]:
# m = folium.Map(location=[28.5212389, 81.0903013], zoom_start=10, tiles='CartoDB positron')

## Map Visualization of the survey

In [35]:
map_vis = folium.Map(location=[28.5212389, 81.0903013], zoom_start=10, tiles="https://mt1.google.com/vt/lyrs=y&x={x}&y={y}&z={z}",
        attr="Google",
        name="Google Satellite")

In [39]:
# Create a geometry list from the GeoDataFrame
geo_df_list = [[point.xy[1][0], point.xy[0][0]] for point in gdf_point.point_geometry]

# Iterate through list and add a marker for each volcano, color-coded by its type.
i = 0
for coordinates in geo_df_list:
    # assign a color marker for the type of volcano, Strato being the most common
    if gdf_point.crop[i] == "wheat":
        type_color = "green"
    elif gdf_point.crop[i] == "Mustard":
        type_color = "blue"
    elif gdf_point.crop[i] == "lentil":
        type_color = "orange"
    elif gdf_point.crop[i] == "vegetable":
        type_color = "pink"
    else:
        type_color = "purple"

    # Place the markers with the popup labels and data
    map_vis.add_child(
        folium.Marker(
            location=coordinates,
            popup=
                "Crop: " + str(gdf_point.crop[i]) + "<br>"
                "Other Crop: " + str(gdf_point.specify_other_crop[i]) + "<br>"
                + "Enumerator: " + str(gdf_point.enumerators[i]) + "<br>"
                + "Remarks: " + str(gdf_point.remarks[i]) + "<br>"
                + "Submission Time" + str(gdf_point.submission_time[i])),
                # icon=folium.Icon(color="%s" % type_color),
        )
#     )
    i = i + 1

In [40]:
for _, r in gdf_poly.iterrows():
    # Without simplifying the representation of each borough,
    # the map might not be displayed
    sim_geo = gpd.GeoSeries(r['geometry']).simplify(tolerance=0.001)
    geo_j = sim_geo.to_json()
    geo_j = folium.GeoJson(data=geo_j,
                           style_function=lambda x: {'fillColor': 'orange'})
    folium.Popup(r['enumerators']).add_to(geo_j)
    geo_j.add_to(map_vis)


## Kobo points and polygon being collected
Visualization of the sample points and polygon overlayed on google hybrid map

In [ ]:

map_vis